# Delete a user

## What it does

Removes a user. It returns no data.

## What happens to the saved users

It creates a practice user and deletes it. The real users stay the same.

## Feature it uses

`deleteUser(id)`, from `logic/delete-user.js`.

## Failures it can show

- A number nobody has → `User not found`.
- The user cannot be deleted → `Could not delete the user`.

Open it with `pnpm jupyter` from `first-app`.

In [1]:
import os
import subprocess
from pathlib import Path

app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent

code = r"""
try {
  const { default: Table } = await import("cli-table3");
  const { createUser } = await import("./logic/create-user.js");
  const { deleteUser } = await import("./logic/delete-user.js");

  const practice = createUser("Practice", `practice-${Date.now()}@x.com`);

  // 1. Delete the user: a table that says it returns no data
  deleteUser(practice.id);
  const table = new Table({
    head: ["result"],
    style: { head: ["cyan", "bold"] },
  });
  table.push(["Returns no data."]);
  console.log(table.toString());

  // 2. Delete it again: "User not found"
  try {
    deleteUser(practice.id);
  } catch (error) {
    console.log(error.message);
  }
} catch (error) {
  console.log(error.message);
}
"""

if (app / "logic").is_dir():
    env = os.environ.copy()
    env["FORCE_COLOR"] = "1"
    env.pop("NO_COLOR", None)
    completed = subprocess.run(
        ["node", "--no-warnings", "--input-type=module", "-e", code],
        cwd=app,
        capture_output=True,
        text=True,
        env=env,
    )
    print(completed.stdout, end="")
    if completed.returncode != 0:
        print(completed.stderr, end="")
else:
    print("Cannot find first-app.")

┌──────────────────┐
│ result           │
├──────────────────┤
│ Returns no data. │
└──────────────────┘
User not found
